In [1]:
import sys; sys.path.append('..')
import glob, os, json
import pandas as pd
from src import settings, TextRetrievalEngine, VisionRetrievalEngine, RAGEngine, EvaluationRunner


c:\Users\kyle0\miniconda3\envs\rag_system\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_files = glob.glob(os.path.join(settings.RAW_DATA_DIR, "*.pdf"))
text_engine = TextRetrievalEngine()
base_retriever, all_docs = text_engine.build_or_load(pdf_files)

vision_retriever = VisionRetrievalEngine().load_index()
engine = RAGEngine(base_retriever, all_docs, vision_retriever)


c:\Users\kyle0\Develops\Financial_RAG_System\notebooks\..\src\retrieval\text_engine.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 30541.42it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  F

[DB Load] 하드디스크에 저장된 DB와 문서를 불러옵니다!


c:\Users\kyle0\Develops\Financial_RAG_System\notebooks\..\src\retrieval\text_engine.py:36: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=db_dir, embedding_function=self.embedding_model)


Verbosity is set to 1 (active). Pass verbose=0 to make quieter.


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 297.42it/s]
The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


You are using in-memory collection. This means every image is stored in memory.
You might want to rethink this if you have a large collection!
Loaded 4654 images from 10 JSON files.


✅ 비전 인덱스 'multi_doc_vision_index' 로드 완료
Neural Reranker 로딩 중... (Device: cuda)


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4996.09it/s]


Reranker 모델 로드 완료: BAAI/bge-reranker-v2-m3


In [3]:
import nltk
import ssl

# Mac/Windows 환경에서 SSL 인증서 에러로 다운로드가 조용히 막히는 것을 방지
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# 필수 토크나이저 데이터 명시적 다운로드 (최신 버전 호환을 위해 punkt_tab 추가)
nltk.download('punkt')
nltk.download('punkt_tab')

print("✅ NLTK 토크나이저 데이터 다운로드 완료! 이제 평가 셀을 다시 실행해 보세요.")

✅ NLTK 토크나이저 데이터 다운로드 완료! 이제 평가 셀을 다시 실행해 보세요.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kyle0\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kyle0\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
eval_data_path = os.path.join("..", "data", "eval_dataset.json")
with open(eval_data_path, 'r', encoding='utf-8') as f:
    eval_dataset = json.load(f)
print(f"✅ {len(eval_dataset)}개의 평가 데이터셋을 로드했습니다.")

✅ 10개의 평가 데이터셋을 로드했습니다.


In [ ]:
runner = EvaluationRunner(engine)
df_results = runner.run_full_evaluation(eval_dataset)

metrics_to_show = ["Exact_Match", "ROUGE-L", "BLEU", "LLM_Judge", "Total_Tokens", "Latency (sec)"]
summary = df_results.groupby("Model")[metrics_to_show].mean()

display(summary.reindex([
    "Method 0 (Baseline)", 
    "Method 1 (Vision Only)", 
    "Method 2 (Dual Basic)",
    "w/o Filter", 
    "w/o Reranker", 
    "w/o Agent Window",
    "SOTA (Full)", 
]))